In [21]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cleaned_delivery.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (43739, 16)


,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys


In [22]:
print(df.info())

print(df.isnull().sum())

print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 43739 entries, 0 to 43738
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         43739 non-null  str    
 1   Agent_Age        43739 non-null  int64  
 2   Agent_Rating     43739 non-null  float64
 3   Store_Latitude   43739 non-null  float64
 4   Store_Longitude  43739 non-null  float64
 5   Drop_Latitude    43739 non-null  float64
 6   Drop_Longitude   43739 non-null  float64
 7   Order_Date       43739 non-null  str    
 8   Order_Time       43739 non-null  str    
 9   Pickup_Time      43739 non-null  str    
 10  Weather          43739 non-null  str    
 11  Traffic          43739 non-null  str    
 12  Vehicle          43739 non-null  str    
 13  Area             43739 non-null  str    
 14  Delivery_Time    43739 non-null  int64  
 15  Category         43739 non-null  str    
dtypes: float64(5), int64(2), str(9)
memory usage: 8.6 MB
None
Order_ID   

In [23]:
def calculate_distance(lat1, lon1, lat2, lon2):

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2

    c = 2 * np.arcsin(np.sqrt(a))

    return 6371 * c


df["Distance"] = calculate_distance(
    df["Store_Latitude"],
    df["Store_Longitude"],
    df["Drop_Latitude"],
    df["Drop_Longitude"]
)

df[["Distance"]].head()

,Distance
0,3.025149
1,20.183530
2,1.552758
3,7.790401
4,6.210138


In [24]:
df["Distance"].describe()

count    43739.000000
mean        38.561752
std        534.564299
min          1.465067
25%          4.663432
50%          9.220450
75%         13.682379
max      19692.674606
Name: Distance, dtype: float64

In [25]:
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    dayfirst=True,
    errors="coerce"
)

order_time = pd.to_datetime(df["Order_Time"], errors="coerce")

df["Order_Hour"] = order_time.dt.hour

df["Day"] = df["Order_Date"].dt.day

df["Day_Of_Week"] = df["Order_Date"].dt.dayofweek

df["Month"] = df["Order_Date"].dt.month

df[["Order_Date", "Order_Time", "Order_Hour", "Day", "Day_Of_Week", "Month"]].head()

C:\Users\divya\AppData\Local\Temp\ipykernel_48528\2813003668.py:1: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Order_Date"] = pd.to_datetime(
C:\Users\divya\AppData\Local\Temp\ipykernel_48528\2813003668.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  order_time = pd.to_datetime(df["Order_Time"], errors="coerce")


,Order_Date,Order_Time,Order_Hour,Day,Day_Of_Week,Month
0,2022-03-19,11:30:00,11.0,19,5,3
1,2022-03-25,19:45:00,19.0,25,4,3
2,2022-03-19,08:30:00,8.0,19,5,3
3,2022-04-05,18:00:00,18.0,5,1,4
4,2022-03-26,13:30:00,13.0,26,5,3


In [26]:
df["Is_Weekend"] = (df["Day_Of_Week"] >= 5).astype(int)

df[["Day_Of_Week", "Is_Weekend"]].head()

,Day_Of_Week,Is_Weekend
0,5,1
1,4,0
2,5,1
3,1,0
4,5,1


In [27]:
df["Is_Peak_Hour"] = df["Order_Hour"].isin(
    [8, 9, 10, 17, 18, 19, 20]
).astype(int)

df[["Order_Hour", "Is_Peak_Hour"]].head()

,Order_Hour,Is_Peak_Hour
0,11.0,0
1,19.0,1
2,8.0,1
3,18.0,1
4,13.0,0


In [28]:
df["Traffic"] = df["Traffic"].astype("string").str.strip()

df["Traffic"] = df["Traffic"].replace(["NaN", "nan", ""], pd.NA)

df["Traffic"] = df["Traffic"].fillna(df["Traffic"].mode()[0])

print(df["Traffic"].value_counts(dropna=False))

Traffic
Low       15090
Jam       13725
Medium    10628
High       4296
Name: count, dtype: int64[pyarrow]


In [29]:
traffic_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Jam": 4
}

df["Distance_Traffic"] = df["Distance"] * df["Traffic"].map(traffic_map)

print("Traffic Missing:", df["Traffic"].isnull().sum())

print("Distance_Traffic Missing:", df["Distance_Traffic"].isnull().sum())

df[["Traffic", "Distance", "Distance_Traffic"]].head()

Traffic Missing: 0
Distance_Traffic Missing: 0


,Traffic,Distance,Distance_Traffic
0,High,3.025149,9.075448
1,Jam,20.183530,80.734119
2,Low,1.552758,1.552758
3,Medium,7.790401,15.580803
4,High,6.210138,18.630414


In [30]:
df["Distance_Category"] = pd.cut(
    df["Distance"],
    bins=[0, 2, 5, 10, 20, float("inf")],
    labels=["Very Short", "Short", "Medium", "Long", "Very Long"],
    include_lowest=True
)

df[["Distance", "Distance_Category"]].head()

,Distance,Distance_Category
0,3.025149,Short
1,20.183530,Very Long
2,1.552758,Very Short
3,7.790401,Medium
4,6.210138,Medium


In [31]:
df["Rating_Category"] = pd.cut(
    df["Agent_Rating"],
    bins=[0, 3, 4, 5],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

df[["Agent_Rating", "Rating_Category"]].head()

,Agent_Rating,Rating_Category
0,4.9,High
1,4.5,High
2,4.4,High
3,4.7,High
4,4.6,High


In [32]:
new_columns = [
    "Distance",
    "Order_Hour",
    "Day",
    "Day_Of_Week",
    "Month",
    "Is_Weekend",
    "Is_Peak_Hour",
    "Distance_Category",
    "Distance_Traffic",
    "Rating_Category"
]

df[new_columns].head(10)

,Distance,Order_Hour,Day,Day_Of_Week,Month,Is_Weekend,Is_Peak_Hour,Distance_Category,Distance_Traffic,Rating_Category
0,3.025149,11.0,19,5,3,1,0,Short,9.075448,High
1,20.183530,19.0,25,4,3,0,1,Very Long,80.734119,High
2,1.552758,8.0,19,5,3,1,1,Very Short,1.552758,High
3,7.790401,18.0,5,1,4,0,1,Medium,15.580803,High
4,6.210138,13.0,26,5,3,1,0,Medium,18.630414,High
5,4.610365,21.0,11,4,3,0,0,Short,18.441461,High
6,16.600361,19.0,4,4,3,0,1,Long,66.401445,High
7,20.205253,17.0,14,0,3,0,1,Very Long,40.410506,High
8,19.975520,20.0,20,6,3,1,1,Long,79.902080,High
9,10.280582,21.0,12,5,2,1,0,Long,41.122328,High


In [33]:
print("Total Rows:", df.shape[0])

print("Total Columns:", df.shape[1])

print("\nMissing Values:")

print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Total Rows: 43739
Total Columns: 26

Missing Values:
Order_ID              0
Agent_Age             0
Agent_Rating          0
Store_Latitude        0
Store_Longitude       0
Drop_Latitude         0
Drop_Longitude        0
Order_Date            0
Order_Time            0
Pickup_Time           0
Weather               0
Traffic               0
Vehicle               0
Area                  0
Delivery_Time         0
Category              0
Distance              0
Order_Hour           91
Day                   0
Day_Of_Week           0
Month                 0
Is_Weekend            0
Is_Peak_Hour          0
Distance_Traffic      0
Distance_Category     0
Rating_Category      53
dtype: int64

Duplicate Rows: 0


In [34]:
missing = df.isnull().sum()

print("Columns with missing values:")
print(missing[missing > 0])

print("\nInvalid Order Time values:")
print(df.loc[df["Order_Hour"].isnull(), "Order_Time"].value_counts(dropna=False))

print("\nAgent Rating values:")
print(df.loc[df["Rating_Category"].isnull(), "Agent_Rating"].value_counts(dropna=False))

print("\nRating range:")
print(df["Agent_Rating"].min())
print(df["Agent_Rating"].max())

Columns with missing values:
Order_Hour         91
Rating_Category    53
dtype: int64

Invalid Order Time values:
Order_Time
NaN     91
Name: count, dtype: int64

Agent Rating values:
Agent_Rating
6.0    53
Name: count, dtype: int64

Rating range:
1.0
6.0


In [35]:
df["Order_Hour"] = df["Order_Hour"].fillna(-1).astype(int)

print("Missing Order_Hour:", df["Order_Hour"].isnull().sum())

Missing Order_Hour: 0


In [36]:
df["Rating_Category"] = pd.cut(
    df["Agent_Rating"],
    bins=[0, 3, 4, 6],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

print(df["Rating_Category"].value_counts(dropna=False))
print("Missing Rating_Category:", df["Rating_Category"].isnull().sum())

Rating_Category
High      41295
Medium     2298
Low         146
Name: count, dtype: int64
Missing Rating_Category: 0


In [37]:
print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])

print("\nMissing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDuplicates:", df.duplicated().sum())

Total Rows: 43739
Total Columns: 26

Missing Values:
Series([], dtype: int64)

Duplicates: 0


In [ ]:
df.to_csv("../data/processed/engineered_delivery.csv", index=False)

print("Dataset Saved Successfully!")
print("Final Shape:", df.shape)

Dataset Saved Successfully!
Final Shape: (43739, 26)
